# Mouse Protein Expression: Data Audit

This notebook explores the UCI Mice Protein Expression dataset before machine-learning modelling.

## Objectives

- Inspect the dataset structure and class labels.
- identify missing protein-expression measurements.
- examine repeated measurements from the same mouse.
- prevent data leakage during model evaluation.

In [6]:
from ucimlrepo import fetch_ucirepo
import pandas as pd

mice = fetch_ucirepo(id=342)

X = mice.data.features
y = mice.data.targets
ids = mice.data.ids

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [16]:
display(
    mice.variables[
        ["name", "role", "type", "missing_values"]
    ].head(10)
)
display(mice.variables[["name", "role"]].tail(10))

,name,role,type,missing_values
0,MouseID,ID,Categorical,no
1,DYRK1A_N,Feature,Continuous,yes
2,ITSN1_N,Feature,Continuous,no
3,BDNF_N,Feature,Continuous,yes
4,NR1_N,Feature,Continuous,no
5,NR2A_N,Feature,Continuous,yes
6,pAKT_N,Feature,Continuous,no
7,pBRAF_N,Feature,Continuous,yes
8,pCAMKII_N,Feature,Continuous,no
9,pCREB_N,Feature,Continuous,yes


,name,role
72,pCFOS_N,Feature
73,SYP_N,Feature
74,H3AcK18_N,Feature
75,EGR1_N,Feature
76,H3MeK4_N,Feature
77,CaNA_N,Feature
78,Genotype,Feature
79,Treatment,Feature
80,Behavior,Feature
81,class,Target


In [5]:
print("Object type:")
print(type(mice))

print("\nMain sections:")
print(mice.keys())

print("\nSections inside mice.data:")
print(mice.data.keys())

Object type:
<class 'ucimlrepo.dotdict.dotdict'>

Main sections:
dict_keys(['data', 'metadata', 'variables'])

Sections inside mice.data:
dict_keys(['ids', 'features', 'targets', 'original', 'headers'])


## 1. Dataset structure

In [2]:
print("Feature-table shape:", X.shape)
print("Target-table shape:", y.shape)

print("\nID columns:")
print(ids.columns.tolist())

print("\nTarget columns:")
print(y.columns.tolist())

display(ids.head())
display(X.head())
display(y.head())

Feature-table shape: (1080, 80)
Target-table shape: (1080, 1)

ID columns:
['MouseID']

Target columns:
['class']


,MouseID
0,309_1
1,309_2
2,309_3
3,309_4
4,309_5


,DYRK1A_N,ITSN1_N,BDNF_N,NR1_N,NR2A_N,pAKT_N,pBRAF_N,pCAMKII_N,pCREB_N,pELK_N,...,pS6_N,pCFOS_N,SYP_N,H3AcK18_N,EGR1_N,H3MeK4_N,CaNA_N,Genotype,Treatment,Behavior
0,0.503644,0.747193,0.430175,2.816329,5.990152,0.218830,0.177565,2.373744,0.232224,1.750936,...,0.106305,0.108336,0.427099,0.114783,0.131790,0.128186,1.675652,Control,Memantine,C/S
1,0.514617,0.689064,0.411770,2.789514,5.685038,0.211636,0.172817,2.292150,0.226972,1.596377,...,0.106592,0.104315,0.441581,0.111974,0.135103,0.131119,1.743610,Control,Memantine,C/S
2,0.509183,0.730247,0.418309,2.687201,5.622059,0.209011,0.175722,2.283337,0.230247,1.561316,...,0.108303,0.106219,0.435777,0.111883,0.133362,0.127431,1.926427,Control,Memantine,C/S
3,0.442107,0.617076,0.358626,2.466947,4.979503,0.222886,0.176463,2.152301,0.207004,1.595086,...,0.103184,0.111262,0.391691,0.130405,0.147444,0.146901,1.700563,Control,Memantine,C/S
4,0.434940,0.617430,0.358802,2.365785,4.718679,0.213106,0.173627,2.134014,0.192158,1.504230,...,0.104784,0.110694,0.434154,0.118481,0.140314,0.148380,1.839730,Control,Memantine,C/S


,class
0,c-CS-m
1,c-CS-m
2,c-CS-m
3,c-CS-m
4,c-CS-m


## 2. Select the protein-expression predictors

The UCI feature table includes genotype, treatment, and behavior alongside the protein-expression measurements. Because the class label is constructed from these three experimental descriptors, using them to predict the class would cause target leakage. Therefore, only the 77 protein-expression variables are retained as model predictors.

In [17]:
metadata_columns = ["Genotype", "Treatment", "Behavior"]

X_proteins = X.drop(columns=metadata_columns)

print("Original feature-table shape:", X.shape)
print("Protein-only feature-table shape:", X_proteins.shape)

Original feature-table shape: (1080, 80)
Protein-only feature-table shape: (1080, 77)


## 3. Define the target and biological grouping variable

The modelling target is the eight-class experimental label. Because each biological mouse contributes 15 measurements, the numerical mouse identifier is extracted from `MouseID`. This grouping variable will later prevent measurements from the same mouse from appearing in both training and testing sets.